# Manuscript results and figures

This notebook reviews the reported aggregate results and recreates the three result figures in the short manuscript. It does not train models or load patient-level data.


## 1. Locate the clean reproduction project

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

HERE = Path.cwd().resolve()
while HERE != HERE.parent and not (HERE / "pyproject.toml").is_file():
    HERE = HERE.parent
ROOT = HERE
PAPER_INPUTS = ROOT / "results" / "runs" / "paper_inputs"
if not PAPER_INPUTS.is_dir():
    raise FileNotFoundError(
        f"Paper inputs missing at {PAPER_INPUTS}; run uv run health-fairness run all first."
    )
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print(f"Project: {ROOT}")
print(f"Analysis inputs: {PAPER_INPUTS}")
print(f"Figures will be saved to: {FIGURES}")


Project: /home/senior/master/multi-attribute-fairness-audit
Analysis inputs: /home/senior/master/multi-attribute-fairness-audit/results/runs/paper_inputs
Figures will be saved to: /home/senior/master/multi-attribute-fairness-audit/figures


## 2. Tables in the short manuscript

The following tables use the stored experiment summaries and follow the order and columns of the results tables in the manuscript. Values are rounded to three decimals for display. All paper-facing metrics use unweighted held-out records.


### Predictive performance before mitigation

Each value is the mean across three evaluations for the indicated classifier and task. Higher ROC-AUC and balanced accuracy (BA) are better.


In [2]:
task_labels = {
    "yale": "Yale ED", "brfss": "BRFSS diabetes", "diabetes130": "Diabetes 130-US",
    "nhanes_hba1c": "NHANES HbA1c", "nhanes_bp": "NHANES BP", "nhanes_obesity": "NHANES BMI",
}
model_labels = {"logistic_regression": "Logistic regression", "random_forest": "Random forest", "xgboost": "XGBoost"}
task_order = list(task_labels)
model_order = list(model_labels)

baseline = pd.read_csv(PAPER_INPUTS / "baseline_performance_by_classifier.csv")
assert len(baseline) == 18 and not baseline.duplicated(["dataset", "model"]).any()
performance = baseline.pivot(index="dataset", columns="model", values=["auc", "ba"])
performance = performance.reindex(index=task_order, columns=pd.MultiIndex.from_product([["auc", "ba"], model_order]))
performance = performance.swaplevel(0, 1, axis=1).reindex(columns=pd.MultiIndex.from_product([model_order, ["auc", "ba"]]))
performance.index = performance.index.map(task_labels)
performance.columns = pd.MultiIndex.from_tuples([(model_labels[model], {"auc": "ROC-AUC ↑", "ba": "BA ↑"}[metric]) for model, metric in performance.columns])
performance.index.name = "Dataset / outcome"
display(performance.round(3))


Logistic regression        Random forest          XGBoost  \
                            ROC-AUC ↑   BA ↑     ROC-AUC ↑   BA ↑ ROC-AUC ↑   
Dataset / outcome                                                             
Yale ED                         0.856  0.769         0.857  0.771     0.855   
BRFSS diabetes                  0.807  0.733         0.802  0.729     0.810   
Diabetes 130-US                 0.634  0.596         0.663  0.611     0.675   
NHANES HbA1c                    0.691  0.636         0.672  0.621     0.666   
NHANES BP                       0.607  0.575         0.609  0.578     0.601   
NHANES BMI                      0.675  0.629         0.673  0.622     0.672   

                          
                    BA ↑  
Dataset / outcome         
Yale ED            0.768  
BRFSS diabetes     0.736  
Diabetes 130-US    0.624  
NHANES HbA1c       0.614  
NHANES BP          0.573  
NHANES BMI         0.616

### Mitigation aimed at intersectional groups

AOD is measured across intersectional groups, and ΔBA is the change in balanced accuracy relative to the baseline. Each value is the mean of three evaluations.


In [3]:
mitigation = pd.read_csv(PAPER_INPUTS / "mitigation_by_classifier.csv")
assert len(mitigation) == 36 and not mitigation.duplicated(["dataset", "model", "method"]).any()
assert set(mitigation["method"]) == {"Reweighing", "EO"}
baseline_check = mitigation.groupby(["dataset", "model"])["baseline_aod"].nunique()
assert baseline_check.eq(1).all()
index = pd.MultiIndex.from_product([task_order, model_order], names=["dataset", "model"])
base_gap = mitigation.groupby(["dataset", "model"])["baseline_aod"].first().reindex(index)
by_method = mitigation.pivot(index=["dataset", "model"], columns="method", values=["aod", "delta_ba"]).reindex(index)
mitigation_table = pd.DataFrame({
    "Baseline AOD ↓": base_gap,
    "Reweighing AOD ↓": by_method[("aod", "Reweighing")],
    "EO AOD ↓": by_method[("aod", "EO")],
    "Reweighing ΔBA ↑": by_method[("delta_ba", "Reweighing")],
    "EO ΔBA ↑": by_method[("delta_ba", "EO")],
})
mitigation_table.index = pd.MultiIndex.from_tuples([(task_labels[d], model_labels[m]) for d, m in mitigation_table.index], names=["Dataset / outcome", "Classifier"])
display(mitigation_table.round(3))


Baseline AOD ↓  Reweighing AOD ↓  \
Dataset / outcome Classifier                                              
Yale ED           Logistic regression           0.120             0.107   
                  Random forest                 0.131             0.119   
                  XGBoost                       0.137             0.129   
BRFSS diabetes    Logistic regression           0.504             0.414   
                  Random forest                 0.489             0.384   
                  XGBoost                       0.522             0.396   
Diabetes 130-US   Logistic regression           0.150             0.127   
                  Random forest                 0.175             0.161   
                  XGBoost                       0.186             0.159   
NHANES HbA1c      Logistic regression           0.690             0.684   
                  Random forest                 0.663             0.654   
                  XGBoost                       0.606             0.600   
NHANES BP         Logistic regression           0.591             0.537   
                  Random forest                 0.571             0.497   
                  XGBoost                       0.473             0.422   
NHANES BMI        Logistic regression           0.420             0.426   
                  Random forest                 0.382             0.385   
                  XGBoost                       0.352             0.353   

                                       EO AOD ↓  Reweighing ΔBA ↑  EO ΔBA ↑  
Dataset / outcome Classifier                                                 
Yale ED           Logistic regression     0.041            -0.001    -0.038  
                  Random forest           0.052            -0.000    -0.023  
                  XGBoost                 0.045            -0.000    -0.040  
BRFSS diabetes    Logistic regression     0.065            -0.004    -0.105  
                  Random forest           0.066            -0.011    -0.114  
                  XGBoost                 0.065            -0.010    -0.111  
Diabetes 130-US   Logistic regression     0.069            -0.002    -0.053  
                  Random forest           0.070             0.003    -0.056  
                  XGBoost                 0.037             0.002    -0.114  
NHANES HbA1c      Logistic regression     0.261            -0.006    -0.082  
                  Random forest           0.279            -0.002    -0.071  
                  XGBoost                 0.261            -0.009    -0.071  
NHANES BP         Logistic regression     0.192            -0.006    -0.066  
                  Random forest           0.147            -0.004    -0.068  
                  XGBoost                 0.191            -0.010    -0.065  
NHANES BMI        Logistic regression     0.175            -0.002    -0.082  
                  Random forest           0.178            -0.002    -0.070  
                  XGBoost                 0.176            -0.001    -0.072

### Selected groups before and after equalized odds

For one logistic-regression evaluation, the manuscript reports the supported groups with the lowest and highest baseline TPR in BRFSS and NHANES HbA1c. Counts and rates use unweighted held-out records.


In [4]:
groups = pd.read_csv(PAPER_INPUTS / "selected_group_examples.csv")
selected = groups[groups["dataset"].isin(["brfss", "nhanes_hba1c"])].copy()
assert selected.groupby("dataset")["seed"].nunique().eq(1).all()
assert selected.groupby("dataset")["model"].nunique().eq(1).all()
extremes = []
for dataset in ["brfss", "nhanes_hba1c"]:
    subset = selected[selected["dataset"].eq(dataset)].sort_values("tpr_baseline")
    extremes.append(subset.iloc[[0, -1]])
group_table = pd.concat(extremes)[["dataset", "group", "n_positive", "n_negative", "tpr_baseline", "fpr_baseline", "tpr_eo", "fpr_eo"]].copy()
group_table["dataset"] = group_table["dataset"].map(task_labels)
group_table = group_table.rename(columns={"dataset": "Dataset", "group": "Recorded group", "n_positive": "n+", "n_negative": "n-", "tpr_baseline": "TPR ↑", "fpr_baseline": "FPR ↓", "tpr_eo": "TPR EO ↑", "fpr_eo": "FPR EO ↓"}).reset_index(drop=True)
display(group_table.round(3))


,Dataset,Recorded group,n+,n-,TPR ↑,FPR ↓,TPR EO ↑,FPR EO ↓
0,BRFSS diabetes,White non-Hispanic × Male × 18-39,110,6105,0.436,0.086,0.644,0.401
1,BRFSS diabetes,Black non-Hispanic × Female × 65+,431,808,0.870,0.618,0.650,0.386
2,NHANES HbA1c,Male | Non-Hispanic Asian | 40-59,34,152,0.118,0.132,0.317,0.312
3,NHANES HbA1c,Female | Mexican American | 60+,42,76,0.857,0.803,0.614,0.564


### NHANES validation-design comparison

The regenerated comparison uses repeated single splits (SS) and five-fold cross-validation with combined predictions (CV). The three-way audit requires at least 30 positive and 30 negative cases per group; AOD is not computable when fewer than two groups are supported.


In [5]:
comparison = pd.read_csv(ROOT / "results" / "runs" / "nhanes_validation_comparison.csv")
assert len(comparison) == 6 and not comparison.duplicated(["task", "design"]).any()
design_labels = {"repeated_single_split_60_20_20": "SS", "five_fold_cv_combined_out_of_fold": "CV"}
comparison["design"] = comparison["design"].map(design_labels)
comparison["mean_three_way_aod_min30"] = pd.to_numeric(comparison["mean_three_way_aod_min30"], errors="coerce")
metrics = {"mean_roc_auc": "ROC-AUC", "mean_supported_three_way_groups_min30": "Groups", "mean_three_way_aod_min30": "AOD"}
validation_table = comparison.pivot(index="task", columns="design", values=list(metrics)).reindex(
    index=["HbA1c", "Blood pressure", "BMI"], columns=pd.MultiIndex.from_product([list(metrics), ["SS", "CV"]]))
validation_table.columns = pd.MultiIndex.from_tuples([(metrics[metric], design) for metric, design in validation_table.columns])
validation_table.index.name = "Task"
display(validation_table.style.format({("ROC-AUC", "SS"): "{:.3f}", ("ROC-AUC", "CV"): "{:.3f}", ("Groups", "SS"): "{:.1f}", ("Groups", "CV"): "{:.1f}", ("AOD", "SS"): "{:.3f}", ("AOD", "CV"): "{:.3f}"}, na_rep="n/a"))


## 3. Results figures


In [6]:
# Recreate the article figures from generated experiment inputs.\n"""Figure reproduction from freshly assembled aggregate outputs."""\nfrom pathlib import Path
import json,hashlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
P=ROOT;F=FIGURES;S=PAPER_INPUTS
OUT=F/"data";OUT.mkdir(exist_ok=True)
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":8.8,"axes.titlesize":10,
 "axes.labelsize":8.5,"pdf.fonttype":42,"svg.fonttype":"none","axes.spines.top":False,
 "axes.spines.right":False,"axes.titlelocation":"left"})
INK="#243746";RW="#0072B2";EO="#B95813";GREEN="#277A61";PURPLE="#76558B"
labels=dict(yale="Yale ED",brfss="BRFSS",diabetes130="Diabetes 130-US",
 nhanes_hba1c="NHANES HbA1c",nhanes_bp="NHANES BP",nhanes_obesity="NHANES BMI")
models=["logistic_regression","random_forest","xgboost"]
marks=dict(zip(models,["o","s","^"]));short=dict(zip(models,["LR","RF","XGB"]))
axes_order=["Sex/gender","Race/ethnicity","Age","Intersection"]
G=pd.read_csv(S/"audit_by_condition.csv")
T=pd.read_csv(S/"mitigation_paired.csv")
E=pd.read_csv(S/"selected_group_examples.csv")
sources=["audit_by_condition.csv","mitigation_paired.csv","selected_group_examples.csv"]
assert len(T)==108
def save(fig,name):
 fig.savefig(F/(name+".png"),dpi=300,facecolor="white")
 display(fig)
 plt.close(fig)
def style(ax):
 ax.grid(color="#D9E0E5",lw=.5,alpha=.6);ax.set_axisbelow(True)
 ax.tick_params(labelsize=8)

# RQ1. Every available main audit axis; no conditional selection by result.
D=G[G.primary&G.method.eq("Baseline")]
H=D.groupby(["dataset","axis"]).aod.mean().unstack("axis").reindex(index=list(labels),columns=axes_order)
N=D.groupby(["dataset","axis"]).aod.count().unstack("axis").reindex_like(H)
H.to_csv(OUT/"fig2_audit_scope_means.csv");N.to_csv(OUT/"fig2_audit_scope_counts.csv")
fig,ax=plt.subplots(figsize=(7.2,3.7))
im=ax.imshow(H.to_numpy(),cmap="Blues",vmin=0,vmax=1,aspect="auto")
for i in range(len(H)):
 for j in range(4):
  v=H.iloc[i,j]
  ax.text(j,i,"—" if pd.isna(v) else f"{v:.3f}",ha="center",va="center",fontsize=10,
    color="white" if v>.58 else INK,fontweight="bold" if j==3 else "normal")
ax.set_xticks(range(4),["Sex/gender","Race/ethnicity","Age","Intersectional groups"])
ax.set_yticks(range(6),list(labels.values()));ax.tick_params(length=0)
ax.xaxis.tick_top()
for spine in ax.spines.values():spine.set_visible(False)
ax.set_xticks(np.arange(-.5,4,1),minor=True);ax.set_yticks(np.arange(-.5,6,1),minor=True)
ax.grid(which="minor",color="white",lw=2);ax.tick_params(which="minor",bottom=False,left=False)
ax.add_patch(Rectangle((2.5,-.5),1,6,fill=False,ec=INK,lw=1.2))
cb=fig.colorbar(im,ax=ax,fraction=.035,pad=.03);cb.set_label("Mean AOD · smaller gap ↓")
fig.text(.5,.035,"Means across 3 classifiers × 3 evaluations. Intersectional groups: race × sex × age for all tasks.",ha="center",fontsize=7.6)
fig.subplots_adjust(left=.23,right=.90,top=.89,bottom=.12);save(fig,"fig2_audit_scope")
# RQ2. All tested BRFSS targets and both interventions, audited on the same three-way grouping.
targets=["Sex/gender","Race/ethnicity","Age","race_x_sex","Intersection"]
targetlabels=["Sex","Race/ethnicity","Age","Race × sex","Race × sex × age"]
Z=G[G.dataset.eq("brfss")&G.primary&G.axis.eq("Intersection")]
base=Z[Z.method.eq("Baseline")].aod.mean()
z=Z[Z.target.isin(targets)&Z.method.isin(["Reweighing","EO"])]
assert z.groupby(["target","method"]).size().eq(9).all()
z.to_csv(OUT/"fig3_target_all_evaluations.csv",index=False)
fig,ax=plt.subplots(figsize=(7.2,3.75))
for i,target in enumerate(targets):
 for j,method,c,mark in [(0,"Reweighing",RW,"o"),(1,"EO",EO,"s")]:
  vals=z[z.target.eq(target)&z.method.eq(method)].aod.to_numpy();y=i+(j-.5)*.22
  ax.scatter(vals,np.full(9,y),color=c,s=10,alpha=.25,marker=mark,zorder=2)
  ax.scatter(vals.mean(),y,color=c,s=46,marker=mark,edgecolor="white",lw=.6,zorder=4)
  ax.text(1.07+j*.19,i,f"{vals.mean():.3f}",transform=ax.get_yaxis_transform(),va="center",ha="center",color=c,fontsize=8.5,weight="bold")
ax.text(1.07,1.04,"RW",transform=ax.transAxes,ha="center",color=RW,fontsize=8,weight="bold")
ax.text(1.26,1.04,"EO",transform=ax.transAxes,ha="center",color=EO,fontsize=8,weight="bold")
ax.axvline(base,ls="--",color=INK,lw=1)
ax.text(base,.99,f"Baseline {base:.3f}",transform=ax.get_xaxis_transform(),ha="center",va="bottom",fontsize=8,color=INK)
ax.set(yticks=range(5),yticklabels=targetlabels,xlim=(0,.67),xlabel="Intersectional AOD after mitigation · smaller gap ←")
ax.invert_yaxis();style(ax)
ax.legend(handles=[Line2D([],[],color=c,marker=m,ls="",label=l) for c,m,l in [(RW,"o","Reweighing"),(EO,"s","Equalized odds")]],loc="lower left",bbox_to_anchor=(0,-.32),ncol=2,frameon=False,fontsize=8)
fig.subplots_adjust(left=.23,right=.81,top=.90,bottom=.24);save(fig,"fig3_target")
# RQ3. Entire tested threshold grid, aligned gap and coverage panels; fixed predictions.
fig,axs=plt.subplots(2,2,figsize=(7.2,4.4),sharex="col")
configs=[("nhanes_hba1c",RW,"o","HbA1c"),("nhanes_bp",GREEN,"s","BP"),("nhanes_obesity",EO,"^","BMI")]
all_support=[]
for ds,c,mark,label in configs:
 z=G[G.dataset.eq(ds)&G.method.eq("Baseline")&G.axis.eq("Intersection")].groupby("minimum")[["aod","groups"]].mean()
 all_support.append(z.reset_index().assign(dataset=ds))
 for ax,var in [(axs[0,0],"aod"),(axs[1,0],"groups")]:ax.plot(z.index,z[var],marker=mark,color=c,label=label,lw=1.6,ms=4)
 if ds=="nhanes_hba1c":
  for cut,offset in [(30,(0,10)),(50,(-10,-17))]:
   axs[0,0].annotate(f"{z.loc[cut,'aod']:.3f}",(cut,z.loc[cut,"aod"]),xytext=offset,textcoords="offset points",ha="center",fontsize=8,weight="bold",color=c)
   axs[1,0].annotate(f"{z.loc[cut,'groups']:.0f}",(cut,z.loc[cut,"groups"]),xytext=(0,-14),textcoords="offset points",ha="center",fontsize=9,weight="bold",color=c)
z=G[G.dataset.eq("diabetes130")&G.method.eq("Baseline")&G.axis.eq("Intersection")].groupby("minimum")[["aod","groups"]].mean()
all_support.append(z.reset_index().assign(dataset="diabetes130"))
for ax,var in [(axs[0,1],"aod"),(axs[1,1],"groups")]:ax.plot(z.index,z[var],marker="D",color=PURPLE,lw=1.6,ms=4)
for cut in z.index:
 axs[0,1].annotate(f"{z.loc[cut,'aod']:.3f}",(cut,z.loc[cut,"aod"]),xytext=(0,8),textcoords="offset points",ha="center",fontsize=8,color=PURPLE)
pd.concat(all_support).to_csv(OUT/"fig6_support_means.csv",index=False)
axs[0,0].set(title="NHANES",ylabel="Mean AOD ↓",ylim=(0,1))
axs[0,1].set(title="Diabetes 130-US",ylim=(0,1))
axs[1,0].set(ylabel="Mean supported groups",ylim=(0,38),xlabel="Minimum cases per class")
axs[1,1].set(ylim=(0,38),xlabel="Minimum cases per class")
for ax in axs[:,0]:
 ax.set_xticks([10,20,30,50]);ax.set_xlim(7,53);ax.axvspan(29,31,color=INK,alpha=.055);ax.axvspan(49,51,color=INK,alpha=.055)
for ax in axs[:,1]:ax.set_xticks([20,100,200]);ax.set_xlim(5,215)
for ax in axs.flat:style(ax)
axs[0,0].legend(loc="upper right",frameon=False,fontsize=7.5)
fig.text(.5,.025,"Predictions are unchanged. The minimum-count rule changes which groups enter the audit.",ha="center",fontsize=8,weight="bold",color=INK)
fig.subplots_adjust(left=.1,right=.98,bottom=.14,top=.93,hspace=.12,wspace=.23);save(fig,"fig6_support")
manifest={"version":"final-manuscript","sources":[{"path":"results/runs/paper_inputs/"+s,"sha256":hashlib.sha256((S/s).read_bytes()).hexdigest()} for s in sources],
 "matplotlib":matplotlib.__version__,"pandas":pd.__version__,"no_model_training":True,
 "averaging":"Figures 2, 3 and 6: three models x three evaluations.",
 "uncertainty":"small markers show evaluations, not independent replicates or confidence intervals",
 "figures":3,"supplementary_figures":0,"formats":["png"],"png_dpi":300,"width_inches":7.2}
(F/"provenance.json").write_text(json.dumps(manifest,indent=2))
print("Three manuscript result figures exported.")


<Figure size 720x370 with 2 Axes>

<Figure size 720x375 with 1 Axes>

<Figure size 720x440 with 4 Axes>

Three manuscript result figures exported.


In [ ]:
# RQ3. Support of the 36 NHANES HbA1c intersectional groups. Every participant is tested once
# under cross-validation, so the full-sample counts are the counts used by the pooled audit.
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
C=pd.read_csv(ROOT/"data"/"processed"/"nhanes"/"survey_data.csv")
C=C[C.eligible_hba1c_high.astype(bool)&C.hba1c_high.notna()]
assert len(C)==8081
cnt=C.groupby(["race_ethnicity","sex","age_group"]).hba1c_high.agg(positive="sum",n="size").reset_index()
cnt["negative"]=cnt.n-cnt.positive
assert len(cnt)==36
cnt.to_csv(OUT/"fig7_hba1c_support_counts.csv",index=False)
races=["Mexican American","Other Hispanic","Non-Hispanic White","Non-Hispanic Black","Non-Hispanic Asian","Other/Multi-racial"]
cols=[(s,a) for s in ["Female","Male"] for a in ["20-39","40-59","60+"]]
def cell(r,s,a,col):
 return int(cnt[(cnt.race_ethnicity==r)&(cnt.sex==s)&(cnt.age_group==a)][col].iloc[0])
Pm=np.array([[cell(r,s,a,"positive") for s,a in cols] for r in races])
Nm=np.array([[cell(r,s,a,"negative") for s,a in cols] for r in races])
assert (Nm>=30).all()  # positives are the binding class in every group
assert (Pm>=30).sum()==17 and (Pm>=50).sum()==6
level=np.where(Pm>=50,2,np.where(Pm>=30,1,0))
fig,ax=plt.subplots(figsize=(7.2,3.3))
ax.imshow(level,cmap=ListedColormap(["#EEF1F4","#9ECAE1","#2C7FB8"]),vmin=0,vmax=2,aspect="auto")
for i in range(len(races)):
 for j in range(len(cols)):
  ax.text(j,i,str(Pm[i,j]),ha="center",va="center",fontsize=10,
    color="white" if level[i,j]==2 else INK,fontweight="bold" if level[i,j]>0 else "normal")
ax.set_xticks(range(6),[f"{s} {a.replace('-','–')}" for s,a in cols],fontsize=8)
ax.set_yticks(range(6),races,fontsize=8.5);ax.tick_params(length=0);ax.xaxis.tick_top()
for spine in ax.spines.values():spine.set_visible(False)
ax.set_xticks(np.arange(-.5,6,1),minor=True);ax.set_yticks(np.arange(-.5,6,1),minor=True)
ax.grid(which="minor",color="white",lw=2);ax.tick_params(which="minor",top=False,left=False)
ax.legend(handles=[Patch(color="#EEF1F4",label="Fewer than 30: excluded"),
 Patch(color="#9ECAE1",label="30–49: included at minimum 30 only"),
 Patch(color="#2C7FB8",label="50 or more: included at 30 and 50")],
 loc="upper center",bbox_to_anchor=(.42,-.03),ncol=3,frameon=False,fontsize=7.4,handlelength=1.2,columnspacing=1.2)
fig.subplots_adjust(left=.22,right=.98,top=.86,bottom=.17);save(fig,"fig7_hba1c_support")
print("HbA1c support figure exported.")
